# Stage 2 — Text Splitting

In this notebook we convert the normalized records created during
document ingestion into LangChain Documents and split them into
retrieval-friendly chunks.

Goals:
- Preserve document metadata and source traceability
- Use LangChain for text splitting
- Keep tables and structured rows intact
- Avoid splitting useful policy sections arbitrarily
- Produce chunks that can later be embedded and stored in a vector database

## Imports

In [2]:
import json
from pathlib import Path
from collections import Counter

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries Imported!")

Libraries Imported!


## Define Paths

In [3]:
PROJECT_PATH = Path.cwd().parent

DATA_DIR = PROJECT_PATH / "data"
PROCESSED_DIR = DATA_DIR / "processed"

NORMALIZED_DIR = PROCESSED_DIR / "normalized"
CHUNKS_DIR = PROCESSED_DIR / "chunks"

NORMALIZED_FILE = NORMALIZED_DIR / "normalized_documents.jsonl"
CHUNKS_FILE = CHUNKS_DIR / "chunks.jsonl"

print("Normalized file:", NORMALIZED_FILE)
print("Chunks file:", CHUNKS_FILE)

Normalized file: /Users/pushkarkamat/Desktop/financial-rag/data/processed/normalized/normalized_documents.jsonl
Chunks file: /Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl


## Load Normalized Records

In [4]:
def load_jsonl(path: Path) -> list[dict]:
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records


normalized_records = load_jsonl(NORMALIZED_FILE)

print(f"Loaded records: {len(normalized_records):,}")

Loaded records: 869


## Inspect Record Types

In [5]:
record_type_counts = Counter(
    record.get("content_type", "unknown")
    for record in normalized_records
)

record_type_counts

Counter({'text': 841, 'table': 28})

## Convert records to LangChain Documents

In [6]:
def record_to_langchain_document(record: dict) -> Document:
    metadata = {
        "document": record.get("document"),
        "file_type": record.get("file_type"),
        "page": record.get("page"),
        "sheet": record.get("sheet"),
        "row_start": record.get("row_start"),
        "row_end": record.get("row_end"),
        "content_type": record.get("content_type"),
        "section": record.get("section"),
        "image_path": record.get("image_path"),
        "source_path": record.get("source_path"),
        "table_id": record.get("table_id"),
    }

    return Document(
        page_content=record.get("text", ""),
        metadata=metadata
    )

### Convert all records

In [7]:
documents = [
    record_to_langchain_document(record)
        for record in normalized_records
]

print(f"Langchain Document: {len(documents):,}")

Langchain Document: 869


In [8]:
documents[0]

Document(metadata={'document': '01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'file_type': 'docx', 'page': None, 'sheet': None, 'row_start': None, 'row_end': None, 'content_type': 'text', 'section': None, 'image_path': None, 'source_path': '/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'table_id': None}, page_content='NORTHSTAR FINANCIAL')

## Create LangChain splitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

## Group related text records

The document ingestion stage intentionally preserves paragraphs, headings,
and tables as separate normalized records.

Before applying character-based splitting, related text records are grouped
into larger logical blocks. Tables and structured records remain independent.

This prevents very short paragraphs from becoming individual embedding chunks.

In [10]:
def group_text_records(documents: list[Document]) -> list[Document]:
    grouped_documents = []

    text_buffer = []
    buffer_metadata = None

    def flush_text_buffer():
        nonlocal text_buffer, buffer_metadata

        if not text_buffer:
            return

        combined_text = "\n\n".join(
            text.strip()
            for text in text_buffer
            if text.strip()
        )

        if combined_text:
            grouped_documents.append(
                Document(
                    page_content=combined_text,
                    metadata=buffer_metadata.copy()
                )
            )

        text_buffer = []
        buffer_metadata = None

    for document in documents:

        content_type = document.metadata.get("content_type")

        # Group text records.
        if content_type == "text":

            current_section = document.metadata.get("section")

            # Start a new text group.
            if not text_buffer:
                text_buffer.append(document.page_content)
                buffer_metadata = document.metadata.copy()
                continue

            previous_section = buffer_metadata.get("section")

            # Continue grouping if the section is the same.
            if current_section == previous_section:
                text_buffer.append(document.page_content)

            else:
                flush_text_buffer()

                text_buffer.append(document.page_content)
                buffer_metadata = document.metadata.copy()

        else:
            # Tables and other record types remain independent.
            flush_text_buffer()
            grouped_documents.append(document)

    # Flush anything remaining.
    flush_text_buffer()

    return grouped_documents

In [11]:
grouped_documents = group_text_records(documents)

print(f"Original records: {len(documents):,}")
print(f"Grouped records: {len(grouped_documents):,}")

Original records: 869
Grouped records: 219


## Split Grouped Documents into Chunks

In [12]:
def split_grouped_document(document: Document) -> list[Document]:
    content_type = document.metadata.get("content_type")

    # Normal policy text
    if content_type == "text":
        return text_splitter.split_documents([document])

    # Tables remain intact
    if content_type == "table":
        return [document]

    # Structured rows remain atomic
    if content_type == "structured_row":
        return [document]

    # Images are handled separately later
    if content_type == "image":
        return []

    # Fallback for any non-empty content
    if document.page_content.strip():
        return text_splitter.split_documents([document])

    return []

## Split grouped documents

In [13]:
chunks = []

for document in grouped_documents:
    document_chunks = split_grouped_document(document)
    chunks.extend(document_chunks)

print(f"Grouped documents: {len(grouped_documents):,}")
print(f"Final chunks: {len(chunks):,}")

Grouped documents: 219
Final chunks: 296


## Assign Chunk Metadata

In [14]:
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{index:08d}"
    chunk.metadata["char_count"] = len(chunk.page_content)

## Inspect Sample Chunks

In [15]:
for i, chunk in enumerate(chunks[:10]):
    print(f"CHUNK {i}")
    print(f"ID: {chunk.metadata.get('chunk_id')}")
    print(f"Type: {chunk.metadata.get('content_type')}")
    print(f"Section: {chunk.metadata.get('section')}")
    print(f"Characters: {len(chunk.page_content)}")
    print()
    print(chunk.page_content)
    print("=" * 100)

CHUNK 0
ID: chunk_00000000
Type: text
Section: None
Characters: 392

NORTHSTAR FINANCIAL

Credit Risk Policy

CRP-001

Version 2.0 | Effective 1 April 2025

Classification: INTERNAL — CONTROLLED

Northstar Financial — Synthetic Internal Document. This document is fictional and created solely for educational and software-development purposes. It does not represent an actual financial institution, regulatory requirement, legal obligation, or financial advice.
CHUNK 1
ID: chunk_00000001
Type: table
Section: Document Control
Characters: 1094

Table 1
Section: Document Control

Row 1: Document ID: Document Title | CRP-001: Credit Risk Policy
Row 2: Document ID: Version | CRP-001: 2.0
Row 3: Document ID: Policy / Procedure Owner | CRP-001: Arvind Menon — Head of Credit Risk
Row 4: Document ID: Business Unit | CRP-001: Credit Risk Department
Row 5: Document ID: Approving Authority | CRP-001: Board Risk Committee
Row 6: Document ID: Effective Date | CRP-001: 1 April 2025
Row 7: Document ID: Pr

## Save the chunks

In [16]:
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for chunk in chunks:
        record = {
            "page_content": chunk.page_content,
            "metadata": chunk.metadata,
        }

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print(f"Saved {len(chunks):,} chunks to:")
print(CHUNKS_FILE)


Saved 296 chunks to:
/Users/pushkarkamat/Desktop/financial-rag/data/processed/chunks/chunks.jsonl


## Validate the saved file

In [17]:
saved_chunks = load_jsonl(CHUNKS_FILE)

print(f"Reloaded chunks: {len(saved_chunks):,}")
assert len(saved_chunks) == len(chunks)

print("Chunk file validation passed.")

Reloaded chunks: 296
Chunk file validation passed.


---

In [18]:
from collections import Counter

document_counts = Counter(
    chunk.metadata.get("document")
    for chunk in chunks
)

for document, count in document_counts.items():
    print(f"{document}: {count} chunks")

print("\nTotal chunks:", len(chunks))

01_Credit_Risk_Policy_CRP-001_v2.0.docx: 79 chunks
02_Credit_Exception_Procedure_CEP-006_v1.3.docx: 64 chunks
03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx: 61 chunks
04_Loan_Origination_Policy_LOP-002_v2.3.docx: 91 chunks
rag_demo_test.pdf: 1 chunks

Total chunks: 296


In [19]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("Document:", chunk.metadata.get("document"))
    print("Page:", chunk.metadata.get("page"))
    print("Section:", chunk.metadata.get("section"))
    print("Content type:", chunk.metadata.get("content_type"))
    print("Characters:", len(chunk.page_content))
    print("Text:")
    print(chunk.page_content[:700])

CHUNK 0
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Page: None
Section: None
Content type: text
Characters: 392
Text:
NORTHSTAR FINANCIAL

Credit Risk Policy

CRP-001

Version 2.0 | Effective 1 April 2025

Classification: INTERNAL — CONTROLLED

Northstar Financial — Synthetic Internal Document. This document is fictional and created solely for educational and software-development purposes. It does not represent an actual financial institution, regulatory requirement, legal obligation, or financial advice.
CHUNK 1
Document: 01_Credit_Risk_Policy_CRP-001_v2.0.docx
Page: None
Section: Document Control
Content type: table
Characters: 1094
Text:
Table 1
Section: Document Control

Row 1: Document ID: Document Title | CRP-001: Credit Risk Policy
Row 2: Document ID: Version | CRP-001: 2.0
Row 3: Document ID: Policy / Procedure Owner | CRP-001: Arvind Menon — Head of Credit Risk
Row 4: Document ID: Business Unit | CRP-001: Credit Risk Department
Row 5: Document ID: Approving Authority | C

In [20]:
for chunk in chunks:
    if chunk.metadata.get("document") == "rag_demo_test.pdf":
        print("Document:", chunk.metadata.get("document"))
        print("Content type:", chunk.metadata.get("content_type"))
        print("Characters:", len(chunk.page_content))
        print("Text:")
        print(chunk.page_content)

Document: rag_demo_test.pdf
Content type: text
Characters: 105
Text:
RAG Demo Test

Loan approval requires verified income, identity documents, and acceptable credit history.


In [21]:
import json
from pathlib import Path

CHUNKS_FILE = Path("../data/processed/chunks/chunks.jsonl")

CHUNKS_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for chunk in chunks:
        record = {
            "text": chunk.page_content,
            "metadata": chunk.metadata,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(chunks):,} chunks to:")
print(CHUNKS_FILE)

Saved 296 chunks to:
../data/processed/chunks/chunks.jsonl


In [22]:
with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    saved_chunks = [json.loads(line) for line in f if line.strip()]

print("Saved chunks:", len(saved_chunks))

print("\nFirst saved chunk:")
print(saved_chunks[0]["text"][:500])

print("\nFirst saved chunk metadata:")
print(saved_chunks[0]["metadata"])

Saved chunks: 296

First saved chunk:
NORTHSTAR FINANCIAL

Credit Risk Policy

CRP-001

Version 2.0 | Effective 1 April 2025

Classification: INTERNAL — CONTROLLED

Northstar Financial — Synthetic Internal Document. This document is fictional and created solely for educational and software-development purposes. It does not represent an actual financial institution, regulatory requirement, legal obligation, or financial advice.

First saved chunk metadata:
{'document': '01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'file_type': 'docx', 'page': None, 'sheet': None, 'row_start': None, 'row_end': None, 'content_type': 'text', 'section': None, 'image_path': None, 'source_path': '/Users/pushkarkamat/Desktop/financial-rag/data/raw/docx/01_Credit_Risk_Policy_CRP-001_v2.0.docx', 'table_id': None, 'chunk_id': 'chunk_00000000', 'char_count': 392}
